# Trabajo final · Data Scientist · Libretta

**Plataforma de facturación por suscripción** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real.

## La pregunta oficial

> **¿A qué clientes contactar este mes para evitar que se vayan?**

**Decisión que habilita:** La lista mensual de clientes a contactar, con capacidad para 300.

**Tipo de problema:** Clasificación (churn) con censura y capacidad limitada.

Las instrucciones completas (qué se entrega, criterios de calidad y defensa) están en el PDF **«Trabajo final · Data Scientist · Instrucciones»**. Este notebook es tu lugar de trabajo: vas de los datos sucios a la recomendación sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 1 | Introducción: definición del problema |
| 2 | Python: limpieza |
| 3 | Estadística |
| 4 | Machine Learning y regresión |
| 5 | Deep Learning |
| 6 | Apache Superset |
| 7 | De modelo a producción |
| 8 | Ciencia de datos aplicada al negocio |
| 9 | Fichas y recomendación |

## 0 · Preparación

Ejecutá esta celda una sola vez. `xgboost` es opcional: si querés usarlo, descomentá la última línea.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib
# %pip install -q xgboost


## 1 · Datos y definición del problema

La celda siguiente **genera las tablas**. Tarda unos segundos. Después mirá qué hay con `inventario()` y `diccionario()`.

Antes de modelar, definí el problema: **tipo de problema, población, período, granularidad y variable objetivo**, y qué información está realmente disponible en el momento de decidir.

In [ ]:
import zlib, base64
TEXTO = (
    "eNqdWduO28gRffdXNBg4INeSTEqjmbF2tUBiY4EFvIMgcR4WgiC0yZaGO7yZl4m9m9l/yWMe8uRPmB/Lqeom2aSkcRzDI166urq6"
    "LqeqmnFa5GUtsiYtPglZiayYiEJmEW7xv4iePYvUXhxUpkpZ7iJZ55VbqTROErme+/PLxUTcqzKS0foHmVTKWz0T+FeKNTjNSjDK"
    "0xk4yCapd2V2aOd634ob0ASB7/vfCvWxxkMRzd7FqapqmRauA+bLqf9quvAdEMdZfIpiMfWvpn5AFPs428mklmcYXU6DpeOxcIaK"
    "WL4g2jrf1SCPFN675SzOanVQZeX6E+F2XKdE780i+amaiBtvIposrtfOG1q6SGQGhuUsvM3jULkb572s4jB3JsIpSr6oD01c5M4W"
    "c6Hf9Wa2mE/E7GKJn/liSzxKFca5VpssS/nJ/a3lshLX0JLhtRLBUj8ZliuxwLP/sCm2Yp+XooCgLJEn4r1gowhFP2D8j1tVKleL"
    "uxa9lJr/MYEW3izIy3hahSottKhhEhcuWTpvsohu1McCOszyMpWJG8ywS3/2ijTmQdk9ZyO8J74Rl1BmMBFzMJ/Jqv5UKBcmgErK"
    "5n2ZD/Ua5qkqoSdSaaXK+xj3FT0cMLPMszyNJT2GeYbnJsRwZqywVxUeZKKYvpJJE1nmmF+TJcgmAdkk6G7INqHEvKEgeXmQWWvi"
    "5n0CQRADZulaRnkJt4lxtda4AL8FW3y51WoM4wazhpz/3Kgsr8Sf4lJL+vrxcxnl73lbf80rWerd/6QQWr/y23dN2KSP/+KN/o2c"
    "lW7eSvGXROr7n2Qp4N3mheWDJFHAP7Rz/7r7uTQ68K+NpH8Q7+UvciXKWFWHXKQqqxroJIqrsFR1zjR1XijsxXUpnKccZjpgxMuX"
    "8J6RdRGfO+Kp/WjfJIkLsaaBXo88OSVPBoQcFIXi/MpgC4dwWMf35Btux+Y74Xvij3iBO5LE64hv9RLat2l4QU7pL9k3/fk1OeET"
    "wRGw0c4FB7x7Sb+XS2/AhyLku7WY0/z5kn79wbjxqfXIfThiThCNvYqWvDJsia8bILx8AArdu4xYx9i28DGaWtDlzVLwvZ3FVZy5"
    "G3IC+LvXa25oIy2TUT00XRp8d288KPUWYQze7RTNxEzu+Hy/FoD7vQpvZcv4KVmxaCo/xmmTdnaekJlfCAumA4KmI0werNE/mE3w"
    "e2zBooKx4LdgAklu5Duw6HcuKw3JFp8srzPpGl0VsqmI1P2dt36sHHI3ncWASLvi/LatfV1iUxf+8c6ipmQOFint/xVT6uAB6Hd7"
    "t2T6neU8Ld5cT63y5FhpCOYn5AR+zC9HUhrMUPcqQ8EA8BFNZUACsF/ZaW7joCKo43K3h2c1pYZvCXgtd0gumK+sN4i5qAlrnVI/"
    "UuWCl0mMPLGL71vkD2PECGjlQecGSgYoP/CqVDRFOZxw64EUs+UABoP2b9s6cqV2pawJ4KCt62vtnYNs51NQLpc62XUhOeeQPE58"
    "NMZDhBTfiwBuPSXyAbUBIQ/yqnssvdnq9Hu/i6MJXdhIfEeKZYqJ+WPKW2DuIHx1DFmOfI9MqCoGHVR3ikx8eUEQ7nWA7fQDmzdb"
    "9kFVRcp4seFwjriD87iH8xsLyqNYVlyN1a7L0m7iLTTBK+B2kDV6ZEJpwxMRtf5KEDrGWaN6pq1teTmipIwDJ9Dv80xV+q3NkNSB"
    "FXvR6N9bI5vl8HMY+mpJNrmziiA3gtTMFNe38ICX4i0cApDaLx2wia/I7HfdKirB2l086nAMrlei8o+XBjsAYrsK5xXPM/w3lb/C"
    "nBd4vWUvvbA3x6FPu+v5ui0caY2TLVnhlLLHPImegYcoNHu/Y5/IFG+6ECF23+gtd1EQLHXAUKKZ69sixjWCmq5oB7qALvK4qvLM"
    "BcfeNHVeG4mzWYVUMHQDjK50QMxkUShdhXI5gZGJoBo/0PGjg8WiAhwoiX0ZVwMlgiozxBRPLW1XniUqcxnBAHbMvlgXrVuyDMYh"
    "AEJQRIY/l19b65+k4JF+XdDwKptjSnrvtSig0ZUbnjco7X4oZarQN8TRLkQViVpv1UKFwyvgWQN5G6P92iAh1jvNVE+kF4y9iC4q"
    "T1ajtAvkJ4XoHXoPLeoz8LYloqo0AMRJD19PoEHW+qaVa8k0fXjqZobwaeyzpszseI1qSAqcjAKGmCF9Dxl2o94w/kP0G5K9WxMj"
    "SsEUdUxGAYg0sudgAMqDqyZmfoiPxXLAifrRiEzrID8nMYo4MQ58MOPJTn7nDOaCsJ1uzydhBgxcH8XqkSQAq5W1fKnIiLDybrwM"
    "rNS6vMuBgw7YqFjXKu+6AoAUvtYFJRIFmq09+5Pz/Ofp89ShMgo5DPMmbE7d3ZKlJkaQ/4mzAZ4hAF4C8Y5WnD6PHK+rxg7HUYGt"
    "TZAnkibNqvXGChE4ONWWpvroGkZIz0N5ZqoNlpqLCqcNJ/3UdUh1HN6pWvt7fcfu/i1h467WeAnLvODryYLga/Lk/xEcpyB2V2uo"
    "ZtbAYdwia1Cfjp7niCvXsMPo2vWiZqO42Zvq4JyBR3a1stoV1VHwE87Pni3Gkbuj7dKynU+TOkN6w5iW9bDnrlXYNvMjP2hrR7pn"
    "9mZoazpo3c63P8GSKks07hrkM5h4HOJuixVj3V56+naf5MhJ3ZFKOTvINJXufEalBOmFs1l9Nw7U/YT2NaHVvVNlzSntLZarE5za"
    "HoCjl9A/VEmrkKekW3i2sk00jOOwvjsbhjpF8ZJQdF7qo5zbvJQVKndI1WjVm7OTJtZWLFSZNrWsMeb2Nd/C3/l+90ctov2Mjd54"
    "LR9a/wtZtOcL/dxwSUFyQgQM0qWDBD5/WbHrtzCyYiy0EukOJX+iACcV5dm0wBCfdVGCpasmlUnLjB/oJR8X0Qu+6VBJC2jX9w5d"
    "HPvUgso/EoiuESuWG3kJ1Oxk50krqznoRqB8lIp1E7U0rY8MVjWdc1cITKdTUcFoCsJqXUPLRt+oa4pPLjy5YSN23SidphLVjHQ2"
    "ai3y97+osDZm22jlbu3mptHFsz5CgdzlrIFrlwDblgXSBm14NMOaIKNoRA4ltMWMc2Lq8GzIOhmyDx0dyoecFNvwCGdJHm76Tn1o"
    "hC2r4AwFe8X26cODYN6eHhA09kxMAT08UTixknEskkP7kk5P86GxWAMXyy51IbApI7i2oMONnfMoFCjhBtMH5tzDLvrtLKrtjB+9"
    "fJ6+fP4zmePkeF8RtJszZPqgy4GXkiQZ+Y8n/im6UTrSYXwiOW7QJra+diJ0B6KOlWKf4pmjQDiKE0yXzvEI2cq5nM59EmoevOCY"
    "xNjAaTtJNAg8uXowDgDrDCuctegREfN1HxHnKEzYaQmipmDfrCQpwt0jH66pppkILcIOjlMrTu/tpx6vi3VMtjF/O+jSb3T24Qv+"
    "qLMAvddOsz3SQJcZGcUNmW0wMnK1nibUoK87LHcTTmgW6iV7b8ETG+M8gC6yUvUuziKFiqPMi/W7sjGnz3REYrq0TgeKekQ11t/F"
    "08vMaeJQWgUXiajcUGafZ1yCOzTPhCrmzHQXfDagThP0EaWDvaltJLDX8H10RUojStPlRW23uT+nz3X0KQ7lqDqc5xG0PEBlt6DE"
    "Znp6RGtIfY31Fk9aryD5uJnojFccTqWdwR6Kg/flICwOM10TnIvBEcEgBOu7Oz4g4PqqEw1vv+wIIDLSAY3w8CVnOEcycgf+LIxF"
    "f3NMdqcahb6yNVVYxgWVbWix9IEsqiGymwmLXVPxYQMfP/CGEK76kJZK07sH/TlZoSbP9DLPnr358fXr0WLDag0PuIv3cUifTOj0"
    "OZGirzNN2ea8/vuP78wgwBaOYKUnU3k50eN/JNG0qcwUdHzlY8dS1vp7zOnqzukejldqaz5zM6769JOAMtAYiA8Npifq8PjZsSvB"
    "9s6uBc3d0b4HRZ7emcqYMdqOKOYJTybp0awijuLHz7QGjz5Muq7jjOFHVjrfgK/4lTCvtHoHmh+M6B59ZW7MtjsJHj9n4/6dVK64"
    "rWMtJTCkMYDV1vfGb8ftDQ7d9+zG2lO3nhVNODprcz40j/8Wt/GvOekW0hf0pXZPnxWc4QFcd0/bvFehqgZiHcfQV4imJw8bsJUW"
    "lVazW+Gjpmxl3rFj1LKM4BjYClPcoyBVzsPDs/8CSzl3vQ=="
)
exec(zlib.decompress(base64.b64decode(TEXTO)).decode())

datos = generar_datos()          # genera las tablas, con una semilla fija: todos los equipos de esta empresa ven lo mismo
globals().update(datos)          # cada tabla queda disponible como variable


In [ ]:
SLUG = "03_Libretta"
import os, sqlite3
import pandas as pd

def inventario():
    """Tablas disponibles: filas, columnas y porcentaje de celdas faltantes."""
    return pd.DataFrame([(n, len(df), len(df.columns), round(float(df.isna().mean().mean() * 100), 1)) for n, df in datos.items()], columns=["tabla", "filas", "columnas", "% de faltantes"])

def diccionario(tabla=None):
    """Qué significa cada columna. Es una descripción neutra: no dice qué está limpio ni cuándo se conoce cada dato."""
    return pd.DataFrame([(t, c, d) for t, cols in DICC.items() if tabla in (None, t) for c, d in cols.items()], columns=["tabla", "columna", "qué es"])

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset", **tablas):
    """Guarda las tablas que pases como archivo SQLite y como CSV, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); con = sqlite3.connect(f"{carpeta}/{SLUG}.db")
    for nombre, df in tablas.items():
        df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False, if_exists="replace")
    con.close(); print(f"Listo: {carpeta}/{SLUG}.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta


In [ ]:
inventario()

In [ ]:
diccionario()

### Tu definición del problema

Completá este diccionario. Si cambia más adelante, cambiá también la respuesta y anotá por qué.

In [ ]:
definicion = {
    "tipo de problema": "",
    "población": "",
    "período": "",
    "granularidad (una fila es…)": "",
    "variable objetivo": "",
    "qué se sabe en el momento de decidir": "",
}
definicion


## 2 · Python

**Qué tenés que resolver en tu caso:** Unificar planes con nombres distintos, resolver cuentas y eventos duplicados, y distinguir pausas, pagos fallidos y bajas.

Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)` y queda en la bitácora de calidad.

In [ ]:
# Tu trabajo


In [ ]:
bitacora_calidad

## 3 · Estadística

**Qué tenés que resolver en tu caso:** Tasa de baja por antigüedad y por plan, con intervalos, y qué cambia si se cuentan o no los clientes recientes.

Mostrá distribuciones e intervalos, y al menos una comparación o prueba con su conclusión y su límite.

In [ ]:
# Tu trabajo


## 4 · Modelado

**Qué tenés que resolver en tu caso:** Variables de uso por ventana; regresión logística, Random Forest y XGBoost; validación temporal; precisión en los 300 contactos.

Empezá por una referencia simple. Validá sin mirar el futuro y elegí la métrica por el costo del error, no por costumbre.

In [ ]:
# Tu trabajo


## 5 · Deep Learning

**Qué tenés que resolver en tu caso:** Una red tabular contra el mejor modelo de árboles: ¿cuándo mejora y cuándo solo agrega costo?

Entrená una red y compará contra la mejor referencia con la misma validación. Decidí si vale su costo y su opacidad.

In [ ]:
# Tu trabajo


## 6 · Superset

**Qué tenés que resolver en tu caso:** Tablero con riesgo por segmento, clientes a contactar y bajas evitadas.

Con `exportar_superset(nombre=tabla, ...)` llevás tus tablas a un archivo SQLite para conectar Apache Superset. Un gráfico por pregunta.

In [ ]:
# Tu trabajo


## 7 · Producción

**Qué tenés que resolver en tu caso:** Contrato de entrada, versión y monitoreo de la deriva del uso, que cambia con los cierres fiscales.

Definí el contrato de entrada (campos, tipos y rangos), la versión, una prueba de humo y las señales de monitoreo con umbral, acción y responsable.

In [ ]:
# Tu trabajo


## 8 · Negocio

**Qué tenés que resolver en tu caso:** El costo de contactar frente al de perder al cliente, y qué decir en cada contacto; explicación sin causalidad.

Decisión, costos, explicación global y local, y una recomendación proporcional a la evidencia.

In [ ]:
# Tu trabajo


## 9 · Fichas y recomendación

### Ficha de evidencia

| Campo | Tu respuesta |
| --- | --- |
| Decisión que se toma | |
| Población y período | |
| Procedimiento | |
| Resultado | |
| Validación | |
| Límite | |

### Ficha de modelo y plan de monitoreo

| Campo | Tu respuesta |
| --- | --- |
| Qué modelo se propone y por qué ese | |
| Con qué datos y qué versión | |
| Contrato de entrada | |
| Riesgos principales | |
| Señales a vigilar, umbral, acción y responsable | |

### Recomendación (una página)

- **Evidencia:** qué muestran los datos.
- **Interpretación:** qué creemos que significa, sin presentarlo como causalidad.
- **Recomendación:** qué conviene hacer, proporcional a la evidencia.
- **Una limitación y un siguiente paso.**